# Extracting Sentences from Reports

In [5]:
import re
import pdfplumber
import spacy
import pandas as pd
from pathlib import Path


nlp = spacy.load("en_core_web_sm")


class FinancialPDFProcessor:

    def __init__(self, pdf_path):
        self.pdf_path = Path(pdf_path)
        self.raw_text = ""
        self.clean_text = ""
        self.sentences = []


    def extract_text(self):
        all_pages = []

        with pdfplumber.open(self.pdf_path) as pdf:
            for page in pdf.pages:
                words = page.extract_words()

                # Sorting solution double column
                words_sorted = sorted(words, key=lambda w: (w['top'], w['x0']))

                page_text = " ".join([w['text'] for w in words_sorted])

                # Delete header and footer 
                filtered_words = [
                    w['text'] for w in words_sorted
                    if 50 < w['top'] < page.height - 50
                ]

                page_text = " ".join(filtered_words)

                # Skip directory
                if self.is_table_of_contents(page_text):
                    continue

                all_pages.append(page_text)

        self.raw_text = "\n".join(all_pages)

 # check 
    def is_table_of_contents(self, text):
        if text.count(".....") > 5:
            return True
        if len(re.findall(r'\b\d+\b', text)) > 50:
            return True
        return False

   # use regex clean text
    def clean_text_content(self):
        text = self.raw_text

        text = re.sub(r'http\S+', '', text)
        text = re.sub(r'\.{3,}', ' ', text)
        text = re.sub(r'\bPage\s+\d+\b', '', text)
        text = re.sub(r'\s+', ' ', text)

        self.clean_text = text.strip()

#Split sentence
    def segment_sentences(self):
        doc = nlp(self.clean_text)

        self.sentences = [
            sent.text.strip()
            for sent in doc.sents
            if len(sent.text.strip()) > 20
        ]

    def run(self):
        self.extract_text()
        self.clean_text_content()
        self.segment_sentences()
        return self.sentences


#batch processing function

def process_folder(input_folder, output_folder="output", merge_all=True):
    input_path = Path(input_folder)
    output_path = Path(output_folder)
    output_path.mkdir(exist_ok=True)

    all_data = []

    pdf_files = list(input_path.glob("*.pdf"))

    print(f"Found {len(pdf_files)} PDF files.")

    for pdf_file in pdf_files:
        print(f"Process··ing: {pdf_file.name}")

        processor = FinancialPDFProcessor(pdf_file)
        sentences = processor.run()

        # save csv
        df_single = pd.DataFrame({
            "source_file": pdf_file.name,
            "sentence_id": range(1, len(sentences) + 1),
            "sentence": sentences
        })

        single_output_path = output_path / f"{pdf_file.stem}_sentences.csv"
        df_single.to_csv(single_output_path, index=False)

        print(f"Saved individual CSV: {single_output_path}")

  
        all_data.append(df_single)

 #merger csv
    if merge_all and all_data:
        merged_df = pd.concat(all_data, ignore_index=True)

        merged_output_path = output_path / "ALL_reports_sentences.csv"
        merged_df.to_csv(merged_output_path, index=False)

        print(f"\nMerged CSV saved: {merged_output_path}")
        print(f"Total sentences: {len(merged_df)}")


#main
if __name__ == "__main__":
    process_folder("Reports", output_folder="Extracted Report Sentences", merge_all=True)

Found 1 PDF files.
Process··ing: financial-stability-report-december-2025.pdf
Saved individual CSV: Extracted Report Sentences\financial-stability-report-december-2025_sentences.csv

Merged CSV saved: Extracted Report Sentences\ALL_reports_sentences.csv
Total sentences: 2028


# Training Causal Model

In [6]:
# TRAINING CAUSAL MODEL
import os
import re
import pickle
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, 
                          AutoConfig, get_linear_schedule_with_warmup)
from torch.optim import AdamW

warnings.filterwarnings("ignore")

# =============================================================================
# CONFIGURATION
# =============================================================================
MODEL_DIR = "Causal Classification Model"
os.makedirs(MODEL_DIR, exist_ok=True)

CONFIG = {
    "model_name": "ProsusAI/finbert",
    "save_dir": MODEL_DIR,
    "thresh_file": os.path.join(MODEL_DIR, "finbert_thresh.pkl"),
    "batch_size": 16,
    "epochs": 3,
    "seed": 42,
    "max_len": 128,
    "oversample_ratio": 0.5,
    "patience": 2
}

CAUSAL_PATTERNS = [
    r'\bdue to\b', r'\bbecause(?: of)?\b', r'\bowing to\b', r'\bcaused by\b',
    r'\bdriven by\b', r'\bstemming from\b', r'\bas a result(?: of)?\b',
    r'\bresulted? (?:in|from)\b', r'\bled to\b', r'\bcontributed to\b',
    r'\btriggered(?: by)?\b', r'\battributed to\b', r'\bin response to\b',
    r'\bfuelled? by\b', r'\bspurred? by\b', r'\bpropelled? by\b',
    r'\bweighed? (?:on|by)\b', r'\bboosted? by\b', r'\bhurt by\b',
    r'\bbenefited? from\b', r'\bhamperd? by\b', r'\bpressured? by\b',
    r'\bsupported? by\b', r'\binduced?\b', r'\bforced?\b', r'\ballowed?\b',
    r'\bprompted?\b', r'\bincentivized?\b', r'\bencouraged?\b',
    r'\bsqueezed? (?:the|its|their)?\b', r'\bsaved? (?:the|its|their)?\b',
    r'\beliminated?\b', r'\bhalt(?:ed|ing)?\b',
    r'\b(?:pushed|drove|forced|sent|propelled) (?:the |its |their |market |investors )?\w+ (?:higher|lower|upward|downward)\b',
    r'\b(?:boosted|increased|decreased|reduced|raised|lowered|slashed|trimmed) (?:the |its |their )?\w+\b',
    r'\b(?:eroded|squeezed|hit|hurt|hampered|stifled|limited|restricted) (?:the |its |their )?\w+\b',
    r'\b(?:stimulated|revived|salvaged|expanded|anchored|supported) (?:the |its |their )?\w+\b',
    r'\b(?:so|thus|thereby|therefore|consequently) \w+ing\b',
    r'\bmaking it (?:more|less) (?:expensive|difficult|easy|profitable)\b',
    r'\bafter (?:the )?(?:company|firm|group|fund|it|they|management|government|bank|fed|regulator|stock|shares?|market|index|bond|currency|oil|commodity)\b',
    r'\bfollowing (?:the |a |an |its |their )?\w+',
    r'\bas \w[\w\s]{0,40}(?:intensified?|accelerated?|appreciated?|depreciated?|tightened?|loosened?|worsened?|eased?|increased?|decreased?|rose|fell|surged?|slumped?|declined?|grew|shrank?|reduced?|slowed?|improved?|soared?|plunged?|mounted?|subsided?|rebounded?|spiked?|collapsed?|contracted?|expanded?)\b',
    r'\blowered? (?:its|their|the) (?:holdings?|position|stake|guidance|forecast|outlook)\b',
    r'\braised? (?:its|their|the) (?:holdings?|position|stake|guidance|forecast|outlook)\b',
    r'\baccretive to\b', r'\bdilutive to\b', r'\bcost synerg\w+\b',
    r'\bmargin (?:expansion|compression|improvement|pressure)\b',
    r'\b(?:revenue|profit|earnings|income|sales) (?:growth|decline|drop|miss|beat)\b',
    r'\bsignall?ed? (?:a |an )?\w+', r'\bexited? (?:its|their|the) position\b',
    r'\bincreased? (?:its|their|the) (?:exposure|weighting|allocation)\b',
    r'\btrimmed? (?:its|their|the) (?:holding|position|stake)\b'
]

def clean_text(text: str) -> str:
    return re.sub(r"\s+", " ", str(text)).strip()

def causal_keyword_score(texts: list) -> np.ndarray:
    scores = []
    for t in texts:
        scores.append(sum(1 for p in CAUSAL_PATTERNS if re.search(p, t.lower())))
    return np.array(scores)

class FinCausalDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        augmented = [f"[KW:{causal_keyword_score([t])[0]}] {t}" for t in texts]
        self.enc = tokenizer(augmented, truncation=True, padding=True, max_length=max_len, return_tensors="pt")
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        return {"input_ids": self.enc["input_ids"][idx], "attention_mask": self.enc["attention_mask"][idx], "labels": self.labels[idx]}

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.85, gamma=2.0):
        super().__init__()
        self.alpha = alpha; self.gamma = gamma
    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, reduction="none")
        pt = torch.exp(-ce)
        a_t = torch.where(targets==1, torch.full_like(ce, self.alpha), torch.full_like(ce, 1-self.alpha))
        return (a_t * (1-pt)**self.gamma * ce).mean()

def train():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    np.random.seed(CONFIG["seed"])
    torch.manual_seed(CONFIG["seed"])

    data_path = os.path.join("Causal Training Dataset", "FinCausal_dataset.parquet")
    if not os.path.exists(data_path):
        print(f"[Error] Training data not found at: {data_path}")
        return

    print(f"\n--- Loading Data: {data_path} ---")
    df = pd.read_parquet(data_path)
    df = df.dropna(subset=["text", "gold"]).copy()
    df["gold"] = pd.to_numeric(df["gold"], errors="coerce")
    df = df.dropna(subset=["gold"])
    df["gold"] = df["gold"].astype(int)
    df["text"] = df["text"].apply(clean_text)

    X_train_raw, X_val, y_train_raw, y_val = train_test_split(
        df["text"].tolist(), df["gold"].tolist(), test_size=0.15, random_state=CONFIG["seed"], stratify=df["gold"].tolist()
    )

    train_df = pd.DataFrame({"text": X_train_raw, "label": y_train_raw})
    noise_df, causal_df = train_df[train_df.label == 0], train_df[train_df.label == 1]
    
    n_causal_target = int(len(noise_df) * CONFIG["oversample_ratio"])
    causal_up = resample(causal_df, replace=True, n_samples=n_causal_target, random_state=CONFIG["seed"])
    train_bal = pd.concat([noise_df, causal_up]).sample(frac=1, random_state=CONFIG["seed"])
    X_train, y_train = train_bal["text"].tolist(), train_bal["label"].tolist()

    print(f"--- Initializing FinBERT on {device} ---")
    tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])
    config = AutoConfig.from_pretrained(CONFIG["model_name"], num_labels=2, hidden_dropout_prob=0.3, attention_probs_dropout_prob=0.3, ignore_mismatched_sizes=True)
    model = AutoModelForSequenceClassification.from_pretrained(CONFIG["model_name"], config=config, ignore_mismatched_sizes=True).to(device)

    train_loader = DataLoader(FinCausalDataset(X_train, y_train, tokenizer, CONFIG["max_len"]), batch_size=CONFIG["batch_size"], shuffle=True)
    val_loader = DataLoader(FinCausalDataset(X_val, y_val, tokenizer, CONFIG["max_len"]), batch_size=CONFIG["batch_size"])

    loss_fn = FocalLoss()
    optimizer = AdamW([{"params": model.bert.parameters(), "lr": 1e-5}, {"params": model.classifier.parameters(), "lr": 5e-5}], weight_decay=0.05)
    total_steps = len(train_loader) * CONFIG["epochs"]
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps * 0.15), num_training_steps=total_steps)

    best_f1, best_state, patience_counter, best_thresh_final = 0, None, 0, 0.5

    print("--- Starting Training Loop ---")
    for epoch in range(1, CONFIG["epochs"] + 1):
        model.train()
        total_loss = 0
        for batch in train_loader:
            optimizer.zero_grad()
            out = model(input_ids=batch["input_ids"].to(device), attention_mask=batch["attention_mask"].to(device))
            loss = loss_fn(out.logits, batch["labels"].to(device))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step()
            total_loss += loss.item()

        model.eval()
        all_probs, all_labels = [], []
        with torch.no_grad():
            for batch in val_loader:
                out = model(input_ids=batch["input_ids"].to(device), attention_mask=batch["attention_mask"].to(device))
                all_probs.extend(torch.softmax(out.logits, dim=-1)[:, 1].cpu().numpy())
                all_labels.extend(batch["labels"].numpy())

        best_t_ep, best_f1_ep = 0.5, 0
        for t in np.arange(0.10, 0.60, 0.05):
            p = (np.array(all_probs) >= t).astype(int)
            f = f1_score(all_labels, p, pos_label=1, zero_division=0)
            if f > best_f1_ep: best_f1_ep, best_t_ep = f, t

        print(f"Epoch {epoch}/{CONFIG['epochs']} | Loss: {total_loss/len(train_loader):.4f} | Val F1: {best_f1_ep:.4f} @ Thr: {best_t_ep:.2f}")

        if best_f1_ep > best_f1:
            best_f1, best_state = best_f1_ep, {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter, best_thresh_final = 0, best_t_ep
        else:
            patience_counter += 1
            if patience_counter >= CONFIG["patience"]: break

    # --- SAVE MODEL ---
    model.load_state_dict(best_state)
    model.save_pretrained(CONFIG["save_dir"])
    tokenizer.save_pretrained(CONFIG["save_dir"])
    with open(CONFIG["thresh_file"], "wb") as f: pickle.dump(best_thresh_final, f)
    print(f"\n--- Model saved successfully to ./{CONFIG['save_dir']} ---")

    # --- FINAL METRICS PRINT OUT ---
    print("\n" + "="*50)
    print("FINAL VALIDATION METRICS (UNSEEN DATA)")
    print("="*50)
    model.eval()
    final_probs, final_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            out = model(input_ids=batch["input_ids"].to(device), attention_mask=batch["attention_mask"].to(device))
            final_probs.extend(torch.softmax(out.logits, dim=-1)[:, 1].cpu().numpy())
            final_labels.extend(batch["labels"].numpy())
            
    final_preds = (np.array(final_probs) >= best_thresh_final).astype(int)
    print(f"Best Optimal Threshold: {best_thresh_final:.2f}\n")
    print("Classification Report:")
    print(classification_report(final_labels, final_preds, target_names=["Noise (0)", "Causal (1)"]))
    print("Confusion Matrix:")
    cm = confusion_matrix(final_labels, final_preds)
    print(f"Actual Noise  : {cm[0][0]} Correct | {cm[0][1]} False Alarms")
    print(f"Actual Causal : {cm[1][1]} Correct | {cm[1][0]} Missed")
    print("="*50 + "\n")

if __name__ == "__main__":
    train()


--- Loading Data: Causal Training Dataset\FinCausal_dataset.parquet ---
--- Initializing FinBERT on cpu ---


Loading weights: 100%|██████████| 201/201 [00:01<00:00, 195.18it/s, Materializing param=classifier.weight]                                      
BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |                                                                                       
-----------------------------+------------+---------------------------------------------------------------------------------------
bert.embeddings.position_ids | UNEXPECTED |                                                                                       
classifier.bias              | MISMATCH   | Reinit due to size mismatch - ckpt: torch.Size([3]) vs model:torch.Size([2])          
classifier.weight            | MISMATCH   | Reinit due to size mismatch - ckpt: torch.Size([3, 768]) vs model:torch.Size([2, 768])

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISMATCH	:ck

--- Starting Training Loop ---
Epoch 1/3 | Loss: 0.0351 | Val F1: 0.3990 @ Thr: 0.55
Epoch 2/3 | Loss: 0.0144 | Val F1: 0.4765 @ Thr: 0.55
Epoch 3/3 | Loss: 0.0107 | Val F1: 0.5102 @ Thr: 0.55


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.15it/s]



--- Model saved successfully to ./Causal Classification Model ---

FINAL VALIDATION METRICS (UNSEEN DATA)
Best Optimal Threshold: 0.55

Classification Report:
              precision    recall  f1-score   support

   Noise (0)       0.99      0.89      0.94      1209
  Causal (1)       0.36      0.87      0.51        86

    accuracy                           0.89      1295
   macro avg       0.68      0.88      0.72      1295
weighted avg       0.95      0.89      0.91      1295

Confusion Matrix:
Actual Noise  : 1076 Correct | 133 False Alarms
Actual Causal : 75 Correct | 11 Missed



# Extracting Causal Sentences from Report Sentences

In [7]:
"""
=============================================================================
SCRIPT 2: RUN CAUSAL PIPELINE (INFERENCE ONLY)
Takes Stage 1 CSV -> Applies Trained Model -> Outputs Stage 3 CSV
=============================================================================
"""

import os
import re
import pickle
import warnings
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

warnings.filterwarnings("ignore")

# =============================================================================
# CONFIGURATION
# =============================================================================
MODEL_DIR = "Causal Classification Model"
THRESH_FILE = os.path.join(MODEL_DIR, "finbert_thresh.pkl")

CAUSAL_PATTERNS = [
    r'\bdue to\b', r'\bbecause(?: of)?\b', r'\bowing to\b', r'\bcaused by\b',
    r'\bdriven by\b', r'\bstemming from\b', r'\bas a result(?: of)?\b',
    r'\bresulted? (?:in|from)\b', r'\bled to\b', r'\bcontributed to\b',
    r'\btriggered(?: by)?\b', r'\battributed to\b', r'\bin response to\b',
    r'\bfuelled? by\b', r'\bspurred? by\b', r'\bpropelled? by\b',
    r'\bweighed? (?:on|by)\b', r'\bboosted? by\b', r'\bhurt by\b',
    r'\bbenefited? from\b', r'\bhamperd? by\b', r'\bpressured? by\b',
    r'\bsupported? by\b', r'\binduced?\b', r'\bforced?\b', r'\ballowed?\b',
    r'\bprompted?\b', r'\bincentivized?\b', r'\bencouraged?\b',
    r'\bsqueezed? (?:the|its|their)?\b', r'\bsaved? (?:the|its|their)?\b',
    r'\beliminated?\b', r'\bhalt(?:ed|ing)?\b',
    r'\b(?:pushed|drove|forced|sent|propelled) (?:the |its |their |market |investors )?\w+ (?:higher|lower|upward|downward)\b',
    r'\b(?:boosted|increased|decreased|reduced|raised|lowered|slashed|trimmed) (?:the |its |their )?\w+\b',
    r'\b(?:eroded|squeezed|hit|hurt|hampered|stifled|limited|restricted) (?:the |its |their )?\w+\b',
    r'\b(?:stimulated|revived|salvaged|expanded|anchored|supported) (?:the |its |their )?\w+\b',
    r'\b(?:so|thus|thereby|therefore|consequently) \w+ing\b',
    r'\bmaking it (?:more|less) (?:expensive|difficult|easy|profitable)\b',
    r'\bafter (?:the )?(?:company|firm|group|fund|it|they|management|government|bank|fed|regulator|stock|shares?|market|index|bond|currency|oil|commodity)\b',
    r'\bfollowing (?:the |a |an |its |their )?\w+',
    r'\bas \w[\w\s]{0,40}(?:intensified?|accelerated?|appreciated?|depreciated?|tightened?|loosened?|worsened?|eased?|increased?|decreased?|rose|fell|surged?|slumped?|declined?|grew|shrank?|reduced?|slowed?|improved?|soared?|plunged?|mounted?|subsided?|rebounded?|spiked?|collapsed?|contracted?|expanded?)\b',
    r'\blowered? (?:its|their|the) (?:holdings?|position|stake|guidance|forecast|outlook)\b',
    r'\braised? (?:its|their|the) (?:holdings?|position|stake|guidance|forecast|outlook)\b',
    r'\baccretive to\b', r'\bdilutive to\b', r'\bcost synerg\w+\b',
    r'\bmargin (?:expansion|compression|improvement|pressure)\b',
    r'\b(?:revenue|profit|earnings|income|sales) (?:growth|decline|drop|miss|beat)\b',
    r'\bsignall?ed? (?:a |an )?\w+', r'\bexited? (?:its|their|the) position\b',
    r'\bincreased? (?:its|their|the) (?:exposure|weighting|allocation)\b',
    r'\btrimmed? (?:its|their|the) (?:holding|position|stake)\b'
]

def clean_text(text: str) -> str:
    return re.sub(r"\s+", " ", str(text)).strip()

def causal_keyword_score(texts: list) -> np.ndarray:
    scores = []
    for t in texts:
        scores.append(sum(1 for p in CAUSAL_PATTERNS if re.search(p, t.lower())))
    return np.array(scores)

class CausalInferencePipeline:
    def __init__(self):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        if not os.path.exists(MODEL_DIR):
            raise FileNotFoundError(f"Missing model directory: '{MODEL_DIR}'. Please run the training script first.")
            
        print(f"Loading Model from '{MODEL_DIR}' onto {self.device}...")
        self.tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
        self.model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(self.device)
        self.model.eval()

        # Load optimal threshold calculated during training
        self.thresh_no_kw = 0.40
        if os.path.exists(THRESH_FILE):
            with open(THRESH_FILE, "rb") as f: 
                self.thresh_no_kw = pickle.load(f)

    def predict(self, sentences: list, thresh_with_kw=0.05) -> pd.DataFrame:
        cleaned = [clean_text(s) for s in sentences]
        kw_scores = causal_keyword_score(cleaned)

        results = pd.DataFrame({"sentence": sentences})
        
        # Format text to give the model the keyword hint it learned during training
        augmented = [f"[KW:{kw}] {t}" for kw, t in zip(kw_scores, cleaned)]
        enc = self.tokenizer(augmented, truncation=True, padding=True, max_length=128, return_tensors="pt")
        
        with torch.no_grad():
            logits = self.model(input_ids=enc["input_ids"].to(self.device), 
                                attention_mask=enc["attention_mask"].to(self.device)).logits
        
        model_probs = torch.softmax(logits, dim=-1).cpu().numpy()[:, 1]
        
        # Dual Threshold Logic: Easier to classify as causal if keywords are present
        thresholds = np.where(kw_scores > 0, thresh_with_kw, self.thresh_no_kw)
        results["is_causal"] = (model_probs >= thresholds).astype(int)
        
        return results

    def process_csv(self, input_csv: str, output_csv: str):
        print(f"\n--- Loading Stage 1 Output: {input_csv} ---")
        if not os.path.exists(input_csv):
            raise FileNotFoundError(f"Input file not found: {input_csv}. Ensure Stage 1 has run.")
            
        df_in = pd.read_csv(input_csv)
        if "sentence" not in df_in.columns:
            raise ValueError("Input CSV must contain a 'sentence' column.")
            
        sentences = df_in["sentence"].dropna().astype(str).tolist()
        
        print(f"Scoring {len(sentences)} extracted sentences...")
        results_df = self.predict(sentences)
        
        causal_df = results_df[results_df["is_causal"] == 1].copy()
        print(f"Gatekeeper complete: Found {len(causal_df)} causal sentences out of {len(sentences)}.")
        
        final_df = pd.DataFrame({
            "id": range(1, len(causal_df) + 1),
            "sentence": causal_df["sentence"].values
        })
        
        os.makedirs(os.path.dirname(output_csv), exist_ok=True)
        final_df.to_csv(output_csv, index=False)
        print(f"--- SUCCESS: Formatted Data saved for Stage 3 at: {output_csv} ---")

if __name__ == "__main__":
    # Define File Paths
    STAGE1_INPUT = "Extracted Report Sentences/ALL_reports_sentences.csv"
    STAGE2_OUTPUT = "causal_output/causal_sentences.csv"
    
    try:
        pipeline = CausalInferencePipeline()
        pipeline.process_csv(input_csv=STAGE1_INPUT, output_csv=STAGE2_OUTPUT)
    except Exception as e:
        print(f"\n[Error] Pipeline Halted: {e}")

Loading Model from 'Causal Classification Model' onto cpu...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 655.37it/s, Materializing param=classifier.weight]                                      



--- Loading Stage 1 Output: Extracted Report Sentences/ALL_reports_sentences.csv ---
Scoring 2028 extracted sentences...
Gatekeeper complete: Found 392 causal sentences out of 2028.
--- SUCCESS: Formatted Data saved for Stage 3 at: causal_output/causal_sentences.csv ---


# JSON conversion in causal form using Gemini

In [1]:
import os
import time
import json
import pandas as pd
from dotenv import load_dotenv
from google import genai
from pydantic import BaseModel, Field

# =============================================================================
# SCHEMA DEFINITIONS (Updated for Batching)
# =============================================================================

class CausalEdge(BaseModel):
    sentence_id: int = Field(description="The exact ID of the sentence this relationship was found in.")
    cause: str = Field(description="The exact phrase representing the cause. Direct quote only.")
    effect: str = Field(description="The exact phrase representing the effect. Direct quote only.")
    polarity: str = Field(description="The direction of the effect: 'positive', 'negative', or 'neutral'.")

class ExtractionResult(BaseModel):
    edges: list[CausalEdge] = Field(description="List of causal relationships found across the batch.")

# =============================================================================
# MAIN EXTRACTOR CLASS
# =============================================================================

class GeminiRelationExtractor:
    def __init__(self):
        load_dotenv()
        api_key = os.getenv("GEMINI_API_KEY")
        
        if not api_key:
            raise ValueError("[Security Error] GEMINI_API_KEY not found in .env file.")
            
        self.client = genai.Client(api_key=api_key)
        self.model_name = 'gemini-3.1-flash-lite-preview'
        self.batch_size = 10  # Optimal size for high-precision extraction
        print("✓ Gemini Client Initialized Securely.")

    def extract_from_batch(self, batch_df: pd.DataFrame, retries: int = 2) -> list[CausalEdge]:
        """Sends a batch of up to 10 sentences to Gemini."""
        
        # 1. Format the batch into a numbered string for the prompt
        sentences_text = ""
        for _, row in batch_df.iterrows():
            sentences_text += f"[ID: {row['id']}] {row['sentence']}\n"
            
        prompt = f"""
        You are a strict causal extraction system for a financial knowledge graph.
        Task: Extract explicit cause-and-effect pairs from the following batch of sentences.
        
        Rules:
        1. 'cause' and 'effect' MUST be exact substrings from the text.
        2. Polarity must be 'positive' (increase/strengthen), 'negative' (decrease/weaken), or 'neutral'.
        3. Match the extracted edges to the correct Sentence ID. If a sentence has no causal link, ignore it.
        
        Sentences to analyze:
        {sentences_text}
        """
        
        for attempt in range(retries + 1):
            try:
                response = self.client.models.generate_content(
                    model=self.model_name, 
                    contents=prompt,
                    config={
                        'response_mime_type': 'application/json',
                        'response_schema': ExtractionResult,
                        'temperature': 0.0, 
                    },
                )
                return response.parsed.edges if response.parsed else []
                
            except Exception as e:
                if "429" in str(e):
                    wait_time = 15 * (attempt + 1)
                    print(f"  [Rate Limit] Gemini quota reached. Backing off for {wait_time}s...")
                    time.sleep(wait_time)
                else:
                    print(f"  [API Error] {e}")
                    break
        return []

    def process_csv_to_json(self, input_csv: str, output_json: str):
        print(f"\n--- Loading Causal Sentences from: {input_csv} ---")
        if not os.path.exists(input_csv):
            raise FileNotFoundError(f"Input file not found: {input_csv}.")
            
        df = pd.read_csv(input_csv)
        all_graph_data = []
        
        total_batches = (len(df) + self.batch_size - 1) // self.batch_size

        print(f"--- Starting Batched Extraction ({len(df)} sentences -> {total_batches} API calls) ---")
        
        # Iterate through the DataFrame in chunks of 10
        for i in range(0, len(df), self.batch_size):
            batch_df = df.iloc[i : i + self.batch_size]
            current_batch_num = (i // self.batch_size) + 1
            
            print(f"Processing Batch {current_batch_num}/{total_batches} (IDs {batch_df['id'].iloc[0]} to {batch_df['id'].iloc[-1]})...")
            
            edges = self.extract_from_batch(batch_df)
            
            for edge in edges:
                # Find the original sentence text using the returned ID to pack into metadata
                try:
                    original_sent = batch_df[batch_df['id'] == edge.sentence_id]['sentence'].values[0]
                except IndexError:
                    original_sent = "Sentence text not found."

                all_graph_data.append({
                    "sentence_id": edge.sentence_id,
                    "source": edge.cause,
                    "target": edge.effect,
                    "metadata": {
                        "polarity": edge.polarity,
                        "original_sentence": original_sent
                    }
                })
                
            # 5-second delay between batches to comfortably clear the 15 RPM free-tier limit
            time.sleep(5)

        # Export to JSON
        os.makedirs(os.path.dirname(output_json), exist_ok=True)
        with open(output_json, 'w', encoding='utf-8') as f:
            json.dump(all_graph_data, f, indent=4)
        
        print(f"\n--- Stage 3 Complete! ---")
        print(f"Successfully extracted {len(all_graph_data)} distinct causal edges.")
        print(f"Data ready for Graph Construction at: {output_json}")

# =============================================================================
# PIPELINE EXECUTION (MAIN)
# =============================================================================

if __name__ == "__main__":
    # Ensure this matches the output from Stage 2
    STAGE2_INPUT = "causal_output/causal_sentences.csv"
    
    STAGE3_OUTPUT = "JSON for Graph creation/financial_graph_edges.json"
    
    try:
        extractor = GeminiRelationExtractor()
        extractor.process_csv_to_json(input_csv=STAGE2_INPUT, output_json=STAGE3_OUTPUT)
    except Exception as e:
        print(f"\n[Pipeline Halted] {e}")

✓ Gemini Client Initialized Securely.

--- Loading Causal Sentences from: causal_output/causal_sentences.csv ---
--- Starting Batched Extraction (392 sentences -> 40 API calls) ---
Processing Batch 1/40 (IDs 1 to 10)...
Processing Batch 2/40 (IDs 11 to 20)...
Processing Batch 3/40 (IDs 21 to 30)...
Processing Batch 4/40 (IDs 31 to 40)...
Processing Batch 5/40 (IDs 41 to 50)...
Processing Batch 6/40 (IDs 51 to 60)...
Processing Batch 7/40 (IDs 61 to 70)...
Processing Batch 8/40 (IDs 71 to 80)...
Processing Batch 9/40 (IDs 81 to 90)...
Processing Batch 10/40 (IDs 91 to 100)...
Processing Batch 11/40 (IDs 101 to 110)...
Processing Batch 12/40 (IDs 111 to 120)...
Processing Batch 13/40 (IDs 121 to 130)...
Processing Batch 14/40 (IDs 131 to 140)...
Processing Batch 15/40 (IDs 141 to 150)...
Processing Batch 16/40 (IDs 151 to 160)...
Processing Batch 17/40 (IDs 161 to 170)...
Processing Batch 18/40 (IDs 171 to 180)...
Processing Batch 19/40 (IDs 181 to 190)...
Processing Batch 20/40 (IDs 191

# Final Graph Creation

In [2]:
import os
import json
import pandas as pd
import networkx as nx
from pyvis.network import Network

# =============================================================================
# 1. FILE PATH CONFIGURATION
# =============================================================================
INPUT_DIR = "JSON for Graph creation"
INPUT_FILE = os.path.join(INPUT_DIR, "financial_graph_edges.json")

OUTPUT_DIR = "Financial Causal Knowledge Graph"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def build_graph():
    # =============================================================================
    # 2. LOAD STAGE 3 JSON DATA
    # =============================================================================
    print(f"[*] Loading data from {INPUT_FILE}...")
    try:
        with open(INPUT_FILE, 'r', encoding='utf-8') as f:
            json_data = json.load(f)
    except FileNotFoundError:
        print(f"[Error] File not found. Please ensure Stage 3 has run successfully.")
        return

    # Flatten the JSON structure into a format pandas can use
    records = []
    for item in json_data:
        records.append({
            "Source": item.get("source"),
            "Target": item.get("target"),
            "Polarity": item.get("metadata", {}).get("polarity", "neutral").lower()
        })

    df_causal = pd.DataFrame(records)
    print(f"[*] Successfully loaded {len(df_causal)} raw causal relationships.")

    # =============================================================================
    # 3. ENTITY RESOLUTION & AGGREGATION
    # =============================================================================
    # Ritam's Synonym Mapping (Expand this dictionary as needed)
    synonym_mapping = {
        "Housing Costs": "House Prices",
        "Jobless Rate": "Unemployment Rate",
        "CPI": "Inflation",
        "Rate hikes": "Interest Rates"
    }

    df_causal['Source'] = df_causal['Source'].replace(synonym_mapping)
    df_causal['Target'] = df_causal['Target'].replace(synonym_mapping)

    # Frequency Aggregation: Count duplicate edges to determine connection strength
    edge_weights = df_causal.groupby(['Source', 'Target', 'Polarity']).size().reset_index(name='Weight')
    print(f"[*] Entity resolution completed. Unique aggregated causal edges: {len(edge_weights)}.")

    # =============================================================================
    # 4. BUILD THE NETWORKX GRAPH
    # =============================================================================
    FCKG = nx.DiGraph()

    for index, row in edge_weights.iterrows():
        source = row['Source']
        target = row['Target']
        relation = row['Polarity']
        weight = row['Weight']
        
        # Color coding based on Gemini's explicit polarity output
        if relation == "positive":
            edge_color = "#2ECC71" # Green (Increases/Drives)
        elif relation == "negative":
            edge_color = "#E74C3C" # Red (Decreases/Inhibits)
        else:
            edge_color = "#95A5A6" # Grey (Neutral)
            
        FCKG.add_edge(
            source, 
            target, 
            label=relation.capitalize(), 
            weight=weight, 
            color=edge_color,
            title=f"Polarity: {relation.capitalize()}<br>Occurrences (Weight): {weight}" 
        )

    # =============================================================================
    # 5. GRAPH ANALYTICS (PageRank)
    # =============================================================================
    # Calculate centrality to find the most influential nodes in the financial system
    pagerank_scores = nx.pagerank(FCKG, weight='weight')

    # Scale the scores visually so more important nodes appear larger
    scaled_pagerank_for_size = {node: score * 600 for node, score in pagerank_scores.items()}
    nx.set_node_attributes(FCKG, scaled_pagerank_for_size, 'value') 
    nx.set_node_attributes(FCKG, pagerank_scores, 'pagerank_score')

    print("\n🏆 --- Top 3 Core Drivers of Systemic Risk (PageRank Centrality) ---")
    sorted_nodes = sorted(pagerank_scores.items(), key=lambda x: x[1], reverse=True)
    for node, score in sorted_nodes[:3]:
        print(f"  -> {node}: {score:.4f}")

    # =============================================================================
    # 6. EXPORT ASSETS (GEXF & HTML)
    # =============================================================================
    # Export for Gephi (Static Analysis)
    gexf_path = os.path.join(OUTPUT_DIR, "Pangura_FCKG_Final.gexf")
    nx.write_gexf(FCKG, gexf_path)
    print(f"\n[+] Data asset exported successfully for Gephi: {gexf_path}")

    # Export for PyVis (Interactive Web Viewer)
    net = Network(height="800px", width="100%", bgcolor="#1e1e1e", font_color="white", directed=True)
    net.from_nx(FCKG)

    # Physics engine settings for optimal financial network display
    net.force_atlas_2based(
        gravity=-50, 
        central_gravity=0.01, 
        spring_length=100, 
        spring_strength=0.08, 
        damping=0.4, 
        overlap=0
    )
    net.show_buttons(filter_=['physics'])

    html_path = os.path.join(OUTPUT_DIR, "Pangura_FCKG_Interactive.html")
    net.save_graph(html_path)
    print(f"[+] Interactive web visualization generated: {html_path}")

if __name__ == "__main__":
    build_graph()

[*] Loading data from JSON for Graph creation\financial_graph_edges.json...
[*] Successfully loaded 302 raw causal relationships.
[*] Entity resolution completed. Unique aggregated causal edges: 298.

🏆 --- Top 3 Core Drivers of Systemic Risk (PageRank Centrality) ---
  -> compounded these effects: 0.0047
  -> capital drawdown: 0.0047
  -> quoted mortgage rates have continued to decrease: 0.0047

[+] Data asset exported successfully for Gephi: Financial Causal Knowledge Graph\Pangura_FCKG_Final.gexf
[+] Interactive web visualization generated: Financial Causal Knowledge Graph\Pangura_FCKG_Interactive.html
